# Verification Notebook V4: Universal Topology

**Claim**: See `paper/manifest.yaml`::R6

**Runtime**: ~1 minute

This notebook verifies universal topology claim: n = 2.00 ± 0.05 across all systems, with Influenza exception.

In [ ]:
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# Load manifest
manifest_path = Path('../paper/manifest.yaml')
manifest = yaml.safe_load(manifest_path.open())
result = manifest['results']['R6']  # R6: Universal n = 2.00 ± 0.05 topology

print(f"Verifying: {result['title']}")
print(f"Result ID: R6")
print(f"Category: topology")

## Load Canonical Data

In [ ]:
# Load data from canonical outputs
data_path = Path('../data/outputs/topology/')

if not data_path.exists():
    print(f"Data directory not found: {data_path}")
    print("   Run: python scripts/populate_outputs.py")
else:
    print(f"Data directory found: {data_path}")
    
# List available files
if data_path.exists():
    files = list(data_path.glob('*'))
    print(f"\nAvailable files:")
    for f in files:
        print(f"  - {f.name}")

## Verify Claims

In [ ]:
# Load topology data
topology_csv = data_path / 'n_per_system.csv'

# Check 1: All systems converge to n ≈ 2.00
print("\nCheck 1: All systems converge to n ≈ 2.00")
if topology_csv.exists():
    topology_df = pd.read_csv(topology_csv)
    # Filter out Influenza pooled (exception)
    topology_df_filtered = topology_df[topology_df['note'] != 'reassortment exception'].copy()
    
    n_values = topology_df_filtered['n'].values
    n_mean = n_values.mean()
    n_std = n_values.std()
    
    expected_mean = 2.00
    expected_uncertainty = 0.05
    
    passed_1 = abs(n_mean - expected_mean) < expected_uncertainty and n_std < expected_uncertainty
    
    print(f"  Mean n: {n_mean:.3f}")
    print(f"  Std n: {n_std:.3f}")
    print(f"  Expected: {expected_mean:.2f} ± {expected_uncertainty:.2f}")
    print(f"  Status: {'PASS' if passed_1 else 'FAIL'}")
    
    print(f"\n  Per-system values:")
    for _, row in topology_df_filtered.iterrows():
        print(f"    {row['system']}: n = {row['n']:.2f} ± {row['std']:.2f}")
else:
    passed_1 = False
    print(f"  Status: FAIL (file not found)")

# Check 2: Influenza exception (reassortment)
print("\nCheck 2: Influenza exception (reassortment)")
if topology_csv.exists():
    topology_df = pd.read_csv(topology_csv)
    influenza_pooled = topology_df[topology_df['system'] == 'Influenza_A_pooled']
    influenza_single = topology_df[topology_df['system'] == 'Influenza_A_single_segment']
    
    if len(influenza_pooled) > 0 and len(influenza_single) > 0:
        n_pooled = influenza_pooled['n'].iloc[0]
        n_single = influenza_single['n'].iloc[0]
        
        # Pooled should be higher due to reassortment
        passed_2 = n_pooled > n_single and n_pooled > 2.1
        
        print(f"  Pooled segments: n = {n_pooled:.2f}")
        print(f"  Single segment: n = {n_single:.2f}")
        print(f"  Difference: {n_pooled - n_single:.2f}")
        print(f"  Expected: pooled > single, pooled > 2.1")
        print(f"  Status: {'PASS' if passed_2 else 'FAIL'}")
    else:
        passed_2 = False
        print(f"  Status: FAIL (data not found)")
else:
    passed_2 = False
    print(f"  Status: FAIL (file not found)")

# Compile results
verified_checks = [
    {'name': 'all_systems_converge', 'expected': 'mean = 2.00 ± 0.05', 'passed': passed_1, 'value': {'mean': float(n_mean), 'std': float(n_std)} if topology_csv.exists() else None},
    {'name': 'influenza_exception', 'expected': 'reassortment → n ≈ 2.2', 'passed': passed_2, 'value': {'pooled': float(n_pooled), 'single': float(n_single)} if topology_csv.exists() and len(influenza_pooled) > 0 else None}
]

all_passed = all(c['passed'] for c in verified_checks)
print(f"\n{'='*60}")
print(f"Overall: {'PASS' if all_passed else 'FAIL'}")
print(f"{'='*60}")

## Update Results

In [ ]:
# Update results.yaml with verification status
results_path = Path('../paper/results.yaml')

if results_path.exists():
    data = yaml.safe_load(results_path.open())
    # Handle both old structure (top-level) and new structure (results key)
    if 'results' in data:
        results = data['results']
    else:
        results = {k: v for k, v in data.items() if k.startswith('R')}
else:
    results = {}

if 'R6' not in results:
    results['R6'] = {}

results['R6']['verified'] = all_passed
results['R6']['verification_date'] = datetime.now().isoformat()
results['R6']['checks'] = verified_checks

# Save updated results with proper structure
output = {'results': results}
with results_path.open('w') as f:
    yaml.dump(output, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"\nResults updated in {results_path}")
print(f"  Verified: {all_passed}")
print(f"  Date: {results['R6']['verification_date']}")